# IK optimization sweep: RR, SR, and platform radius

This notebook tests all requested geometry combinations while keeping the base fixed.

- RR height: `0.5` to `6.0` mm, jump `0.5` mm
- SR height: `1.5` to `7.0` mm, jump `0.5` mm
- Platform radius: `4.0` to `13.0` mm, jump `0.5` mm

Acceptance uses the largest connected all-legs-valid workspace:

- width X `>= 10 mm`
- depth Y `>= 10 mm`
- height Z `> 3 mm`

For every tested combination, the notebook saves:

1. XY, XZ, and YZ workspace projections for `top`, `right`, `left`, and `all legs`
2. 3D workspace view for `top`, `right`, `left`, and `all legs`
3. Angle distributions
4. Angle deviation-from-mean plots
5. Relative-error boxplots and mean/max relative-error plots

In [ ]:
from __future__ import annotations

import json
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "kinematics" / "unified_ik_starter.py").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing kinematics/unified_ik_starter.py")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis.IK_optimizetion.ik_optimization_utils import (
    config_label,
    evaluate_workspace,
    inclusive_range,
    save_full_plot_set,
    save_overview_plots,
)

IK_DIR = REPO_ROOT / "analysis" / "IK_optimizetion"
RESULTS_ROOT = IK_DIR / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print("Repository root:", REPO_ROOT)
print("Results root:", RESULTS_ROOT)

## Configuration

The solver uses effective lengths `d1`, `d2`, and `d3`. This notebook scales those effective lengths proportionally around the current physical design while leaving base anchors unchanged.

Current design mapping:

- platform radius `6.0 mm` -> current `d1`
- SR height `1.5 mm` -> current `d2`
- RR height `3.5 mm` -> current `d3`

The full plot stage is intentionally slow because it saves eight plot files plus a summary JSON for every tested combination.

In [ ]:
# Self-contained bootstrap so this configuration cell can run after a kernel restart.
import json
import math
import sys
from datetime import datetime
from pathlib import Path

import numpy as np

if "REPO_ROOT" not in globals():
    def find_repo_root(start: Path) -> Path:
        start = start.resolve()
        for path in [start, *start.parents]:
            if (path / "kinematics" / "unified_ik_starter.py").exists():
                return path
        raise FileNotFoundError("Could not find repo root containing kinematics/unified_ik_starter.py")
    REPO_ROOT = find_repo_root(Path.cwd())
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

if "inclusive_range" not in globals():
    from analysis.IK_optimizetion.ik_optimization_utils import inclusive_range

if "RESULTS_ROOT" not in globals():
    IK_DIR = REPO_ROOT / "analysis" / "IK_optimizetion"
    RESULTS_ROOT = IK_DIR / "results"
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# ---- Physical sweep ranges requested by the user: 0.5 mm jumps every time ----
RR_MIN_MM, RR_MAX_MM, RR_STEP_MM = 0.5, 6.0, 0.5
SR_MIN_MM, SR_MAX_MM, SR_STEP_MM = 1.5, 7.0, 0.5
PLATFORM_MIN_MM, PLATFORM_MAX_MM, PLATFORM_STEP_MM = 4.0, 13.0, 0.5

# ---- Current/chosen physical values ----
CURRENT_RR_MM = 3.5
CURRENT_SR_MM = 1.5
CURRENT_PLATFORM_RADIUS_MM = 6.0

# ---- Real deterministic workspace grid ----
# Increase XY_POINTS/Z_POINTS for finer results; this will also make every plot denser and slower.
XY_LIMIT_MM = 18.0
Z_MIN_MM = 0.0
Z_MAX_MM = 18.0
XY_POINTS = 21
Z_POINTS = 11
PHI1_RAD = math.pi / 2.0

# ---- Full plot generation ----
GENERATE_ALL_COMBINATION_PLOTS = True
MAX_FULL_PLOT_CONFIGS = None  # Keep None for all combinations; set an integer only for debugging.
PLOT_DPI = 150

MIN_WORKSPACE_WIDTH_MM = 10.0
MIN_WORKSPACE_DEPTH_MM = 10.0
MIN_HEIGHT_SPAN_MM = 3.0

rr_values = inclusive_range(RR_MIN_MM, RR_MAX_MM, RR_STEP_MM)
sr_values = inclusive_range(SR_MIN_MM, SR_MAX_MM, SR_STEP_MM)
platform_values = inclusive_range(PLATFORM_MIN_MM, PLATFORM_MAX_MM, PLATFORM_STEP_MM)
assert np.allclose(np.diff(rr_values), 0.5)
assert np.allclose(np.diff(sr_values), 0.5)
assert np.allclose(np.diff(platform_values), 0.5)
N_CONFIGS = len(rr_values) * len(sr_values) * len(platform_values)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ID = (
    f"run_{timestamp}_{N_CONFIGS}configs_"
    f"rr{RR_MIN_MM:.1f}-{RR_MAX_MM:.1f}_"
    f"sr{SR_MIN_MM:.1f}-{SR_MAX_MM:.1f}_"
    f"platform{PLATFORM_MIN_MM:.1f}-{PLATFORM_MAX_MM:.1f}"
).replace(".", "p")
RUN_DIR = RESULTS_ROOT / RUN_ID
TABLE_DIR = RUN_DIR / "tables"
PLOT_DIR = RUN_DIR / "plots"
OVERVIEW_DIR = PLOT_DIR / "overview"
ALL_CONFIG_DIR = PLOT_DIR / "all_configurations"
for path in [TABLE_DIR, OVERVIEW_DIR, ALL_CONFIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

config = {
    "rr_range_mm": [RR_MIN_MM, RR_MAX_MM, RR_STEP_MM],
    "sr_range_mm": [SR_MIN_MM, SR_MAX_MM, SR_STEP_MM],
    "platform_radius_range_mm": [PLATFORM_MIN_MM, PLATFORM_MAX_MM, PLATFORM_STEP_MM],
    "number_of_combinations": int(N_CONFIGS),
    "current_values_mm": {"rr": CURRENT_RR_MM, "sr": CURRENT_SR_MM, "platform_radius": CURRENT_PLATFORM_RADIUS_MM},
    "workspace_probe": {
        "xy_limit_mm": XY_LIMIT_MM,
        "z_min_mm": Z_MIN_MM,
        "z_max_mm": Z_MAX_MM,
        "xy_points": XY_POINTS,
        "z_points": Z_POINTS,
        "phi1_rad": PHI1_RAD,
    },
    "plot_generation": {
        "generate_all_combination_plots": GENERATE_ALL_COMBINATION_PLOTS,
        "max_full_plot_configs": MAX_FULL_PLOT_CONFIGS,
        "plot_dpi": PLOT_DPI,
    },
    "acceptance": {
        "min_workspace_width_mm": MIN_WORKSPACE_WIDTH_MM,
        "min_workspace_depth_mm": MIN_WORKSPACE_DEPTH_MM,
        "min_height_span_mm": MIN_HEIGHT_SPAN_MM,
    },
}
(RUN_DIR / "run_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print("Run directory:", RUN_DIR)
print(f"All combinations to test: {N_CONFIGS:,}")
print(f"RR values ({len(rr_values)}):", rr_values)
print(f"SR values ({len(sr_values)}):", sr_values)
print(f"Platform values ({len(platform_values)}):", platform_values)

## Run the complete parameter sweep

This evaluates all possible combinations from the requested 0.5 mm jumps. With the configured ranges, that is `12 * 12 * 19 = 2736` geometries.

In [ ]:
print(f"Testing all {N_CONFIGS:,} configurations...")
rows = []
t0 = time.perf_counter()
for idx, (rr, sr, platform_radius) in enumerate((r, s, p) for r in rr_values for s in sr_values for p in platform_values):
    rows.append(evaluate_workspace(rr, sr, platform_radius, config, keep_detail=False))
    if (idx + 1) % 50 == 0 or idx + 1 == N_CONFIGS:
        elapsed = time.perf_counter() - t0
        rate = (idx + 1) / elapsed if elapsed > 0 else float("nan")
        remaining = (N_CONFIGS - idx - 1) / rate if rate > 0 else float("nan")
        print(f"  {idx + 1:>5,}/{N_CONFIGS:,} done  elapsed={elapsed:,.1f}s  est_remaining={remaining:,.1f}s")

results = pd.DataFrame(rows).sort_values(["accepted", "score"], ascending=[False, False]).reset_index(drop=True)
# Recreate output folders in case the folder was deleted after the configuration cell ran.
TABLE_DIR.mkdir(parents=True, exist_ok=True)
results_path = TABLE_DIR / "sweep_results.csv"
results.to_csv(results_path, index=False)
print("Saved:", results_path)
display(results.head(10))

In [ ]:
def nearest_row(df, rr, sr, platform_radius):
    distance = (df["rr_mm"] - rr).abs() + (df["sr_mm"] - sr).abs() + (df["platform_radius_mm"] - platform_radius).abs()
    return df.loc[distance.idxmin()]

current_row = nearest_row(results, CURRENT_RR_MM, CURRENT_SR_MM, CURRENT_PLATFORM_RADIUS_MM)
accepted = results[results["accepted"]].copy()
suggestions = accepted.copy() if not accepted.empty else results.head(50).copy()

# Recreate output folders in case the folder was deleted after the configuration cell ran.
TABLE_DIR.mkdir(parents=True, exist_ok=True)
suggestions_path = TABLE_DIR / "suggested_values_all.csv"
suggestions.to_csv(suggestions_path, index=False)
current_path = TABLE_DIR / "current_values_comparison.csv"
pd.DataFrame([current_row.to_dict()]).to_csv(current_path, index=False)

print(f"Accepted suggestion count: {len(accepted):,}")
print(f"Saved all suggested values: {suggestions_path}")
print("Current/chosen value row:")
display(pd.DataFrame([current_row]))
print("Top suggestions:")
display(suggestions[[
    "rr_mm", "sr_mm", "platform_radius_mm",
    "d1_effective_platform_length", "d2_effective_sr_length", "d3_effective_rr_length",
    "workspace_width_mm", "workspace_depth_mm", "height_span_mm",
    "valid_fraction", "largest_component_fraction", "volume_proxy_mm3", "score", "accepted"
]].head(20))

In [ ]:
OVERVIEW_DIR.mkdir(parents=True, exist_ok=True)
save_overview_plots(results, current_row, suggestions, config, OVERVIEW_DIR)
print("Overview plots saved under:", OVERVIEW_DIR)

## Save full graph set for every tested combination

Each configuration gets a folder:

`plots/all_configurations/rr_<value>/sr_<value>/platform_<value>/`

Each folder contains:

- `workspace_projection_xy.png`
- `workspace_projection_xz.png`
- `workspace_projection_yz.png`
- `workspace_3d_per_leg_and_all.png`
- `angle_distributions.png`
- `angle_deviation_from_mean.png`
- `relative_error_boxplot.png`
- `relative_error_mean_max.png`
- `workspace_summary.json`

This step is intentionally long because it generates the complete requested graphs for every tested variable combination.

In [ ]:
# Recreate output folders in case the folder was deleted after the configuration cell ran.
TABLE_DIR.mkdir(parents=True, exist_ok=True)
ALL_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
if GENERATE_ALL_COMBINATION_PLOTS:
    plot_rows = results.copy()
    if MAX_FULL_PLOT_CONFIGS is not None:
        plot_rows = plot_rows.head(int(MAX_FULL_PLOT_CONFIGS)).copy()
    print(f"Saving full graph set for {len(plot_rows):,} / {len(results):,} tested combinations...")
else:
    plot_rows = pd.concat([pd.DataFrame([current_row]), suggestions.head(10)], ignore_index=True).drop_duplicates(
        subset=["rr_mm", "sr_mm", "platform_radius_mm"]
    )
    print(f"Saving debug graph set for {len(plot_rows):,} current/top-suggestion combinations...")

saved_dirs = []
t0 = time.perf_counter()
for i, (_, row) in enumerate(plot_rows.iterrows(), start=1):
    saved_dirs.append(save_full_plot_set(row, config, ALL_CONFIG_DIR))
    if i % 10 == 0 or i == len(plot_rows):
        elapsed = time.perf_counter() - t0
        rate = i / elapsed if elapsed > 0 else float("nan")
        remaining = (len(plot_rows) - i) / rate if rate > 0 else float("nan")
        print(f"  plots {i:>5,}/{len(plot_rows):,} saved  elapsed={elapsed:,.1f}s  est_remaining={remaining:,.1f}s")

plot_manifest = pd.DataFrame({"plot_dir": [str(p) for p in saved_dirs]})
plot_manifest_path = TABLE_DIR / "plot_manifest.csv"
plot_manifest.to_csv(plot_manifest_path, index=False)
print("Saved plot manifest:", plot_manifest_path)

## Final recommendation list

The final box prints the current design and the ranked suggestions. It explicitly prints the **length of the suggested list** and the physical/effective lengths for every printed suggestion.

**Important modeling warning:** recommendations are valid for the proportional effective-length mapping (`platform/SR/RR -> d1/d2/d3`) used here. Confirm this mapping against CAD or measurements before final fabrication decisions.

In [ ]:
print("RESULTS FOLDER")
print(RUN_DIR)
print()
print("COMBINATION COUNT")
print(f"All tested combinations: {len(results):,}")
print(f"Accepted/suggested combinations: {len(accepted):,}")
print(f"Suggestions table length: {len(suggestions):,}")
print(f"Full plot-set folders saved: {len(saved_dirs) if 'saved_dirs' in globals() else 0:,}")

print()
print("CURRENT / CHOSEN VALUES")
print(
    f"RR physical length={current_row['rr_mm']:.2f} mm, "
    f"SR physical length={current_row['sr_mm']:.2f} mm, "
    f"platform radius length={current_row['platform_radius_mm']:.2f} mm | "
    f"effective solver lengths: d1/platform={current_row['d1_effective_platform_length']:.2f}, "
    f"d2/SR={current_row['d2_effective_sr_length']:.2f}, "
    f"d3/RR={current_row['d3_effective_rr_length']:.2f} | "
    f"workspace={current_row['workspace_width_mm']:.2f} x {current_row['workspace_depth_mm']:.2f} mm, "
    f"height={current_row['height_span_mm']:.2f} mm, "
    f"accepted={bool(current_row['accepted'])}, score={current_row['score']:.2f}"
)

if accepted.empty:
    print()
    print("No candidate met width >= 10 mm, depth >= 10 mm, and height > 3 mm.")
    print("Best non-accepted candidates from this sweep:")
else:
    print()
    print("SUGGESTED VALUES (accepted candidates, ranked by score)")
    print(f"Number/length of suggested list: {len(suggestions):,}")

for rank, (_, row) in enumerate(suggestions.head(30).iterrows(), start=1):
    print(
        f"{rank:02d}. RR physical length={row['rr_mm']:.2f} mm, "
        f"SR physical length={row['sr_mm']:.2f} mm, "
        f"platform radius length={row['platform_radius_mm']:.2f} mm | "
        f"effective lengths d1/platform={row['d1_effective_platform_length']:.2f}, "
        f"d2/SR={row['d2_effective_sr_length']:.2f}, "
        f"d3/RR={row['d3_effective_rr_length']:.2f} | "
        f"workspace={row['workspace_width_mm']:.2f} x {row['workspace_depth_mm']:.2f} mm, "
        f"height={row['height_span_mm']:.2f} mm, "
        f"valid_fraction={row['valid_fraction']:.3f}, "
        f"largest_component_fraction={row['largest_component_fraction']:.3f}, "
        f"volume_proxy={row['volume_proxy_mm3']:.1f}, "
        f"score={row['score']:.2f}"
    )
